# Road Network Quality Index (RNQI)

This notebook evaluates and ranks city road networks using connectivity, efficiency, and hierarchy-based indicators.


## 1. Setup and imports
This cell loads the Python libraries required for geospatial processing, graph analysis, and file handling.


In [1]:
import os
import time
import random

import numpy as np
import pandas as pd
import networkx as nx
import geopandas as gpd
import osmnx as ox

from shapely.geometry import Point

/Users/ankita/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


## 2. Reference city locations
This cell defines the benchmark cities and their latitude/longitude coordinates for comparison.


In [2]:
reference_places = {
    "Chandigarh": {
        "lat": 30.7333,
        "lon": 76.7794,
        "label": "Chandigarh city center",
    },

    "Jaipur": {
        "lat": 26.9124,
        "lon": 75.7873,
        "label": "Jaipur city center",
    },

    "Pune": {
        "lat": 18.5204,
        "lon": 73.8567,
        "label": "Pune city center",
    },

    "Kolkata": {
        "lat": 22.5726,
        "lon": 88.3639,
        "label": "Kolkata city center",
    },

    "Ranipet": {
        "lat": 12.9276,
        "lon": 79.3333,
        "label": "Ranipet city center",
    },
}

## 3. Study area settings
This cell sets the analysis radius, buffer distance, cache folder, and output file for all city evaluations.


In [3]:
STUDY_RADIUS_METERS = 2000
BUFFER_METERS = 250

CACHE_DIR = (
    f"rnqi_radius_{STUDY_RADIUS_METERS}m_"
    f"buffer_{BUFFER_METERS}m"
)

os.makedirs(CACHE_DIR, exist_ok=True)

OUTPUT_PATH = "rnqi_reference_city_metrics.csv"

## 4. Overpass graph download
This cell defines a failover mechanism to download road network data from multiple OpenStreetMap endpoints.


In [ ]:
OVERPASS_ENDPOINTS = [
    "https://overpass-api.de/api",
    "https://overpass.kumi.systems/api",
    "https://overpass.nchc.org.tw/api",
]


def download_graph_with_failover(
    boundary_polygon,
    retries_per_endpoint=1,
):
    """Download road network with Overpass failover."""

    original_endpoint = ox.settings.overpass_url
    last_error = None

    ox.settings.requests_timeout = 600

    ox.settings.overpass_settings = (
        "[out:json][timeout:600][maxsize:1073741824]"
    )

    try:
        for endpoint in OVERPASS_ENDPOINTS:

            ox.settings.overpass_url = endpoint

            for attempt in range(
                1,
                retries_per_endpoint + 1,
            ):

                try:
                    print(
                        f"Querying {endpoint} "
                        f"(attempt {attempt})"
                    )

                    graph = ox.graph_from_polygon(
                        boundary_polygon,
                        network_type="drive",
                        simplify=True,
                    )

                    print("Download successful.")

                    return graph

                except Exception as error:

                    last_error = error

                    print(
                        f"Failed: {type(error).__name__}: {error}"
                    )

                    if attempt < retries_per_endpoint:
                        time.sleep(5)

    finally:
        ox.settings.overpass_url = original_endpoint

    raise RuntimeError(
        f"All Overpass endpoints failed. "
        f"Last error: {last_error}"
    )


## 5. Study-area geometry
This cell creates the circular analysis polygon and converts it between geographic and projected coordinate systems.


In [5]:
def create_study_area(
    lat,
    lon,
    radius_meters,
):
    """
    Create circular study area.

    Returns:
        study_polygon_wgs84
        study_polygon_projected
        projected_crs
    """

    point_gdf = gpd.GeoDataFrame(
        geometry=[Point(lon, lat)],
        crs="EPSG:4326",
    )

    point_projected = ox.projection.project_gdf(
        point_gdf
    )

    projected_crs = point_projected.crs

    study_polygon_projected = (
        point_projected.geometry.iloc[0]
        .buffer(radius_meters)
    )

    study_polygon_wgs84 = (
        gpd.GeoSeries(
            [study_polygon_projected],
            crs=projected_crs,
        )
        .to_crs("EPSG:4326")
        .iloc[0]
    )

    return (
        study_polygon_wgs84,
        study_polygon_projected,
        projected_crs,
    )

## 6. Download or load city road graph
This cell fetches the road network for each city, caches it locally, and clips it to the actual study area.


In [6]:
def get_city_graph(
    city,
    place_info,
):
    """
    Create study area, download or load road graph.
    """

    lat = place_info["lat"]
    lon = place_info["lon"]

    (
        study_polygon,
        study_polygon_projected,
        projected_crs,
    ) = create_study_area(
        lat,
        lon,
        STUDY_RADIUS_METERS,
    )

    area_sq_km = (
        study_polygon_projected.area / 1_000_000
    )

    buffered_polygon_projected = (
        study_polygon_projected.buffer(
            BUFFER_METERS
        )
    )

    buffered_polygon = (
        gpd.GeoSeries(
            [buffered_polygon_projected],
            crs=projected_crs,
        )
        .to_crs("EPSG:4326")
        .iloc[0]
    )

    graph_path = os.path.join(
        CACHE_DIR,
        f"{city.lower()}_drive.graphml",
    )

    if os.path.exists(graph_path):

        print(f"Loading cached graph: {graph_path}")

        graph = ox.load_graphml(graph_path)

    else:

        print("Downloading buffered road network...")

        graph = download_graph_with_failover(
            buffered_polygon
        )

        ox.save_graphml(
            graph,
            graph_path,
        )

    # Clip to actual study area
    graph = ox.truncate.truncate_graph_polygon(
        graph,
        study_polygon,
    )

    if graph.number_of_nodes() == 0:
        raise RuntimeError(
            "No nodes found inside study area."
        )

    graph_projected = ox.project_graph(graph)

    return (
        graph_projected,
        area_sq_km,
        study_polygon,
    )

## 7. Connectivity metrics
This cell computes node density, link density, degree, and sampled closeness to capture network connectivity.


In [7]:
def calculate_connectivity_metrics(
    graph,
    area_sq_km,
    seed=42,
    closeness_sample_size=20,
):
    """
    Calculate Connectivity pillar metrics.
    """

    node_count = graph.number_of_nodes()

    if node_count == 0:
        raise RuntimeError("Graph has no nodes.")

    _, edges = ox.graph_to_gdfs(graph)

    edge_count = graph.number_of_edges()

    road_length_km = (
        edges["length"].sum() / 1000
    )

    average_degree = (
        sum(dict(graph.degree()).values())
        / node_count
    )

    # Sampled closeness
    sample_size = min(
        closeness_sample_size,
        node_count,
    )

    rng = random.Random(seed)

    sample_nodes = rng.sample(
        list(graph.nodes()),
        sample_size,
    )

    closeness_values = []

    for node in sample_nodes:

        lengths = (
            nx.single_source_dijkstra_path_length(
                graph,
                node,
                weight="length",
            )
        )

        reachable_lengths = [
            distance
            for target, distance in lengths.items()
            if target != node
        ]

        if reachable_lengths:

            closeness_values.append(
                len(reachable_lengths)
                / sum(reachable_lengths)
            )

    return {
        "Nodes": node_count,
        "Edges": edge_count,
        "Node Density": (
            node_count / area_sq_km
        ),
        "Link Density": (
            road_length_km / area_sq_km
        ),
        "Average Degree": average_degree,
        "Mean Sampled Closeness": (
            np.mean(closeness_values)
            if closeness_values
            else np.nan
        ),
        "Total Road Length (km)": road_length_km,
    }

## 8. Efficiency metrics
This cell estimates network efficiency using shortest-path lengths, inverse-efficiency values, and circuity.


In [8]:
def calculate_efficiency_metrics(
    graph,
    sample_size=100,
    seed=42,
):
    """
    Calculate Efficiency pillar metrics.

    Uses a sampled approximation for large graphs.
    """

    nodes = list(graph.nodes())

    if len(nodes) < 2:
        raise RuntimeError(
            "At least 2 nodes required."
        )

    rng = random.Random(seed)

    sample_nodes = rng.sample(
        nodes,
        min(sample_size, len(nodes)),
    )

    shortest_path_lengths = []
    efficiency_values = []
    circuity_values = []

    for source in sample_nodes:

        lengths = (
            nx.single_source_dijkstra_path_length(
                graph,
                source,
                weight="length",
            )
        )

        for target, distance in lengths.items():

            if target == source:
                continue

            if distance > 0:

                shortest_path_lengths.append(
                    distance
                )

                efficiency_values.append(
                    1 / distance
                )

                # Euclidean distance
                x1 = graph.nodes[source]["x"]
                y1 = graph.nodes[source]["y"]

                x2 = graph.nodes[target]["x"]
                y2 = graph.nodes[target]["y"]

                euclidean_distance = np.sqrt(
                    (x2 - x1) ** 2
                    + (y2 - y1) ** 2
                )

                if euclidean_distance > 0:

                    circuity_values.append(
                        distance
                        / euclidean_distance
                    )

    return {
        "Global Efficiency": (
            np.mean(efficiency_values)
            if efficiency_values
            else np.nan
        ),

        "ASP Length (m)": (
            np.mean(shortest_path_lengths)
            if shortest_path_lengths
            else np.nan
        ),

        "Circuity": (
            np.mean(circuity_values)
            if circuity_values
            else np.nan
        ),

        "Efficiency Sample Size": len(
            shortest_path_lengths
        ),
    }

## 9. Hierarchy metrics
This cell measures betweenness centrality and road class distribution to assess structural hierarchy.


In [9]:
def calculate_hierarchy_metrics(
    graph,
    sample_size=100,
):
    """
    Calculate Hierarchy pillar metrics.
    """

    node_count = graph.number_of_nodes()

    k = min(
        sample_size,
        node_count,
    )

    betweenness = nx.betweenness_centrality(
        graph,
        k=k,
        weight="length",
        normalized=True,
        seed=42,
    )

    mean_betweenness = np.mean(
        list(betweenness.values())
    )

    # Extract road classes
    _, edges = ox.graph_to_gdfs(graph)

    road_class_counts = {}

    if "highway" in edges.columns:

        for highway_value in edges["highway"].dropna():

            if isinstance(highway_value, list):
                highway_value = highway_value[0]

            road_class_counts[highway_value] = (
                road_class_counts.get(
                    highway_value,
                    0,
                ) + 1
            )

    return {
        "Mean Betweenness": mean_betweenness,
        "Road Class Distribution": str(
            road_class_counts
        ),
    }

## 10. Hierarchy clarity placeholder
This cell defines a preliminary indicator for road-class diversity and leaves room for a formal hierarchy-clarity formula.


In [10]:
def calculate_hierarchy_clarity(edges):
    """
    Placeholder for hierarchy clarity.

    Needs a formally defined formula before
    inclusion in final RNQI.
    """

    if "highway" not in edges.columns:
        return np.nan

    valid_classes = []

    for value in edges["highway"].dropna():

        if isinstance(value, list):
            value = value[0]

        valid_classes.append(value)

    if not valid_classes:
        return np.nan

    unique_classes = len(
        set(valid_classes)
    )

    return unique_classes

## 11. Compute all city metrics
This cell runs the full metric pipeline for each city and compiles the raw RNQI values into a result dictionary.


In [11]:
def calculate_city_metrics(
    city,
    place_info,
):
    """
    Calculate raw RNQI metrics for one city.
    """

    print(f"\nCalculating {city}...")

    graph, area_sq_km, study_polygon = (
        get_city_graph(
            city,
            place_info,
        )
    )

    # Keep largest weakly connected component
    largest_component = max(
        nx.weakly_connected_components(graph),
        key=len,
    )

    analysis_graph = graph.subgraph(
        largest_component
    ).copy()

    connectivity = calculate_connectivity_metrics(
        analysis_graph,
        area_sq_km,
    )

    efficiency = calculate_efficiency_metrics(
        analysis_graph,
    )

    hierarchy = calculate_hierarchy_metrics(
        analysis_graph,
    )

    return {
        "City": city,
        "Study Area (km²)": area_sq_km,
        **connectivity,
        **efficiency,
        **hierarchy,
    }

## 12. Run city-by-city analysis
This cell iterates through all reference cities, executes the metric calculations, and saves each result to the output CSV.


In [12]:
city_results = []
city_errors = []

for city, place_info in reference_places.items():

    try:

        result = calculate_city_metrics(
            city,
            place_info,
        )

        city_results.append(result)

        pd.DataFrame(city_results).to_csv(
            OUTPUT_PATH,
            index=False,
        )

        print(
            f"Saved {len(city_results)}/"
            f"{len(reference_places)} cities"
        )

    except Exception as error:

        city_errors.append(
            {
                "City": city,
                "Error": str(error),
            }
        )

        print(
            f"Skipped {city}: {error}"
        )


Calculating Chandigarh...
Querying https://overpass-api.de/api (attempt 1)
Download successful.
Saved 1/5 cities

Calculating Jaipur...
Querying https://overpass-api.de/api (attempt 1)
Download successful.
Saved 2/5 cities

Calculating Pune...
Querying https://overpass-api.de/api (attempt 1)
Download successful.
Saved 3/5 cities

Calculating Kolkata...
Querying https://overpass-api.de/api (attempt 1)
Download successful.
Saved 4/5 cities

Calculating Ranipet...
Querying https://overpass-api.de/api (attempt 1)
Download successful.
Saved 5/5 cities


## 13. Normalization helpers
This cell defines min-max scaling functions used to convert raw metric values into comparable scores.


In [13]:
def minmax_normalize(series):
    min_value = series.min()
    max_value = series.max()

    if max_value == min_value:
        return pd.Series(
            50.0,
            index=series.index,
        )

    return (
        (series - min_value)
        / (max_value - min_value)
        * 100
    )

## 14. Inverse normalization helper
This cell creates the inverse min-max function so lower-is-better indicators like circuity and ASP are scored positively.


In [14]:
def minmax_normalize_inverse(series):
    min_value = series.min()
    max_value = series.max()

    if max_value == min_value:
        return pd.Series(
            50.0,
            index=series.index,
        )

    return (
        (max_value - series)
        / (max_value - min_value)
        * 100
    )

## 15. Convert metrics to scores
This cell normalizes each raw metric into a 0–100 score so all pillars can be compared fairly.


In [15]:
results_df = pd.DataFrame(city_results)

# Connectivity
results_df["Node Density Score"] = (
    minmax_normalize(
        results_df["Node Density"]
    )
)

results_df["Link Density Score"] = (
    minmax_normalize(
        results_df["Link Density"]
    )
)

results_df["Average Degree Score"] = (
    minmax_normalize(
        results_df["Average Degree"]
    )
)

results_df["Closeness Score"] = (
    minmax_normalize(
        results_df["Mean Sampled Closeness"]
    )
)

# Efficiency
results_df["Global Efficiency Score"] = (
    minmax_normalize(
        results_df["Global Efficiency"]
    )
)

results_df["ASP Score"] = (
    minmax_normalize_inverse(
        results_df["ASP Length (m)"]
    )
)

results_df["Circuity Score"] = (
    minmax_normalize_inverse(
        results_df["Circuity"]
    )
)

# Hierarchy
results_df["Betweenness Score"] = (
    minmax_normalize(
        results_df["Mean Betweenness"]
    )
)

## 16. Pillar score grouping
This cell defines the metrics that belong to connectivity, efficiency, and hierarchy before aggregating each pillar score.


In [16]:
connectivity_columns = [
    "Node Density Score",
    "Link Density Score",
    "Average Degree Score",
    "Closeness Score",
]

efficiency_columns = [
    "Global Efficiency Score",
    "ASP Score",
    "Circuity Score",
]

hierarchy_columns = [
    "Betweenness Score",
]

## 17. Aggregate pillar scores
This cell combines the normalized sub-scores for connectivity, efficiency, and hierarchy into separate pillar totals.


In [17]:
results_df["Connectivity Score"] = (
    results_df[connectivity_columns].mean(axis=1)
)

results_df["Efficiency Score"] = (
    results_df[efficiency_columns].mean(axis=1)
)

results_df["Hierarchy Score"] = (
    results_df[hierarchy_columns].mean(axis=1)
)

## 18. Final RNQI calculation
This cell computes the final composite Road Network Quality Index by averaging the three pillar scores.


In [18]:
results_df["RNQI Score"] = (
    results_df["Connectivity Score"]
    + results_df["Efficiency Score"]
    + results_df["Hierarchy Score"]
) / 3

## 19. Save and display ranking
This cell writes the final scores to CSV and presents the city ranking summary for the final RNQI comparison.


In [19]:
results_df.to_csv(
    "rnqi_final_scores.csv",
    index=False,
)

results_df[
    [
        "City",
        "Connectivity Score",
        "Efficiency Score",
        "Hierarchy Score",
        "RNQI Score",
    ]
].round(2)

,City,Connectivity Score,Efficiency Score,Hierarchy Score,RNQI Score
0,Chandigarh,75.41,0.42,19.94,31.93
1,Jaipur,69.98,13.70,0.00,27.89
2,Pune,43.79,90.32,18.06,50.72
3,Kolkata,46.88,81.08,23.46,50.47
4,Ranipet,37.44,80.75,100.00,72.73
